# 03c · Build the generation prompts (no API key needed)

For every English question this notebook runs each system's retrieval and builds the **exact prompt** that system
would send to its language model, then writes everything to one file:
`03_rag/answers/generation_inputs.jsonl` (one line per question x system).

| ID | System | Retrieval | Prompt |
|---|---|---|---|
| L0 | LLM, no retrieval | none | PlantIQ's advisory prompt with no passages |
| SK | Selva Kumar et al. | LangChain dense top-4 (RecursiveCharacterTextSplitter 1000/200, FAISS) | LangChain's default QA prompt |
| PQ | PlantIQ | full hybrid search, top-3 (dense + BM25, RRF, rerank, topic filter, MMR) | PlantIQ's advisory prompt |

**This tests the RAG, not the language model:** one model answers every prompt, so differences come from retrieval,
prompting and grounding. The answers and their scores are produced from this file (see the README) and stored in
`03_rag/answers/answers.jsonl` and `03_rag/answers/judgments.jsonl`; notebook 03d turns them into results.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "03_rag"))

import json
import os

import pandas as pd

from common import config, env
from common import rag_eval as R
from plantiq_rag_snapshot.advisory import (CONFIDENCE_THRESHOLD_RAG, SYSTEM_PROMPT, build_prompt, display_name,
                                           retrieve)
from plantiq_rag_snapshot.build import build_retriever, load_embedder
from plantiq_rag_snapshot.profiles import BALANCED, QUALITY

config.ensure_dirs()
device = env.require_cuda().type
OUT_DIR = config.ROOT / ("_smoke/03_rag/answers" if config.SMOKE else "03_rag/answers")
OUT_DIR.mkdir(parents=True, exist_ok=True)
qs = pd.read_json(config.RAG_DATA / "questions.jsonl", lines=True)
qs = qs[(qs.lang == "en") & (qs.out_of_kb == 0)].reset_index(drop=True)
print(len(qs), "English questions")

PROFILE = BALANCED if config.SMOKE else QUALITY
SK_EMB = "hf:sentence-transformers/all-MiniLM-L6-v2" if config.SMOKE else \
    os.getenv("SK_EMBEDDINGS", "hf:sentence-transformers/all-mpnet-base-v2")
embedder = load_embedder(PROFILE, device)
rag = build_retriever(PROFILE, config.KB_DIR, config.RAG_INDEX / f"pq_{PROFILE.name}", device=device, embedder=embedder)
sk = R.SKRetriever(SK_EMB, config.KB_DIR, config.RAG_INDEX, device=device)

In [ ]:
rows = []
for _, q in qs.iterrows():
    label = display_name(q["disease"])
    # L0: PlantIQ's prompt, no passages
    rows.append({"key": f"{q['id']}|L0", "id": q["id"], "system": "L0", "system_prompt": SYSTEM_PROMPT,
                 "prompt": build_prompt(q["disease"], 100.0, None, q["question"], "en", [], False), "contexts": []})
    # SK: LangChain dense top-4 + LangChain's default QA prompt
    hits = sk.search(f"{label}: {q['question']}", k=4)
    ctx = [h["text"] for h in hits]
    rows.append({"key": f"{q['id']}|SK", "id": q["id"], "system": "SK", "system_prompt": "",
                 "prompt": R.SK_PROMPT.format(context="\n\n".join(ctx), question=f"{label}: {q['question']}"),
                 "contexts": ctx, "sources": [f"{h['source']} p.{h['page']}" for h in hits]})
    # PQ: PlantIQ's retrieval and prompt (router offline; leaf diagnosis given with 100 % confidence)
    rag._cache.clear()
    docs, info, target = retrieve(disease=q["disease"], question=q["question"], rag=rag, llm=None, language="en")
    top = max((d["score"] for d in docs), default=0.0)
    blended = q["disease"] != "Healthy" and top < CONFIDENCE_THRESHOLD_RAG
    rows.append({"key": f"{q['id']}|PQ", "id": q["id"], "system": "PQ", "system_prompt": SYSTEM_PROMPT,
                 "prompt": build_prompt(q["disease"], 100.0, None, q["question"], target, docs, blended),
                 "contexts": [d["text"] for d in docs],
                 "sources": [f"{d['source']} p.{d['metadata'].get('page')}" for d in docs], "blended": blended})

inputs = OUT_DIR / "generation_inputs.jsonl"
with inputs.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
env.write_json(OUT_DIR / "generation_inputs_meta.json",
               {"n": len(rows), "questions": int(len(qs)), "profile": PROFILE.name, "sk_embeddings": SK_EMB,
                "meta": env.run_meta()})
print(f"wrote {inputs} ({len(rows)} prompts). Next: answers + judgments (README §2, step 12), then notebook 03d.")
pd.DataFrame(rows).groupby("system")["contexts"].apply(lambda c: sum(map(len, c)) / len(c)).rename("avg passages")